# 03 - Qualidade e tratamento dos dados

**Objetivo:** Criar flags de qualidade e bases preservando os valores originais.

> Execute as células na ordem. As saídas são gravadas nas pastas do projeto.

In [ ]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Localiza a raiz do projeto independentemente de o notebook ser aberto pela pasta raiz ou notebooks.
CANDIDATOS=[Path.cwd(), Path.cwd().parent]
ROOT=next((p.resolve() for p in CANDIDATOS if (p/'dados').exists() and (p/'notebooks').exists()),None)
if ROOT is None: raise FileNotFoundError('Abra o VS Code na pasta projeto_soft_sensor.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.funcoes import *
sns.set_theme(style='whitegrid')
print('Raiz:',ROOT)


## Carregamento e preservação

In [ ]:
d=carregar_csv(ROOT/'dados/02_intermediarios/dados_sincronizados.csv'); d['temperatura_interna_original_C']=d.temperatura_interna_C; d['umidade_interna_original_pct']=d.umidade_relativa_interna_pct

## Flags automáticas

In [ ]:
d['flag_temp_ausente']=d.temperatura_interna_C.isna().astype(int); d['flag_ur_ausente']=d.umidade_relativa_interna_pct.isna().astype(int)
d['flag_temp_limite_fisico']=((d.temperatura_interna_C < -20)|(d.temperatura_interna_C > 80)).astype(int)
d['flag_ur_limite_fisico']=((d.umidade_relativa_interna_pct < 0)|(d.umidade_relativa_interna_pct > 100)).astype(int)
d['delta_temp_abs']=d.temperatura_interna_C.diff().abs(); saud=d.evento_referencia.isin(['normal','onda_de_calor_real']); limiar=d.loc[saud,'delta_temp_abs'].quantile(.999)
d['flag_salto_estatistico']=(d.delta_temp_abs>limiar).astype(int); print('Limiar empírico de salto:',limiar); display(d.filter(regex='^flag_').sum().to_frame('registros'))

## Qualidade consolidada

In [ ]:
d['qualidade_consolidada']='valido'
d.loc[d.flag_falha_comunicacao.eq(1),'qualidade_consolidada']='falha_comunicacao'
d.loc[d.flag_dado_faltante_aleatorio.eq(1),'qualidade_consolidada']='faltante_aleatorio'
d.loc[d.flag_temp_limite_fisico.eq(1)|d.flag_ur_limite_fisico.eq(1),'qualidade_consolidada']='fisicamente_impossivel'
d.loc[d.evento_referencia.eq('sensor_travado_temperatura'),'qualidade_consolidada']='sensor_travado'
d.loc[d.evento_referencia.eq('ruido_excessivo_temperatura'),'qualidade_consolidada']='ruido_excessivo'
d.loc[d.evento_referencia.eq('pico_isolado_desconhecido'),'qualidade_consolidada']='pico_desconhecido'
d.loc[d.evento_referencia.eq('onda_de_calor_real'),'qualidade_consolidada']='evento_extremo_real'
display(d.qualidade_consolidada.value_counts().to_frame('registros'))

## Colunas tratadas e exportação

In [ ]:
d['temperatura_interna_tratada_C']=d.temperatura_interna_C.mask(d.flag_temp_limite_fisico.eq(1)); d['umidade_interna_tratada_pct']=d.umidade_relativa_interna_pct.mask(d.flag_ur_limite_fisico.eq(1))
d['apto_treino_soft_sensor']=d.evento_referencia.isin(['normal','onda_de_calor_real']).astype(int)
out=ROOT/'dados/02_intermediarios/dados_com_flags_qualidade.csv'; d.to_csv(out,index=False,encoding='utf-8-sig'); print('Salvo:',out)
plt.figure(figsize=(10,4)); d.qualidade_consolidada.value_counts().plot.bar(); plt.ylabel('Registros'); plt.tight_layout(); plt.savefig(ROOT/'resultados/figuras/qualidade/qualidade_consolidada.png',dpi=150); plt.show()

## Conclusões da etapa

Execute todas as células, registre as observações exibidas e confirme os arquivos gerados antes de avançar para o notebook seguinte.